In [1]:
# =========================
# 环境初始化
# =========================

from typing import TypedDict

from langgraph.graph import StateGraph, END

from openai import OpenAI
from dotenv import load_dotenv

import os

print("环境初始化完成")

环境初始化完成


In [2]:
# =========================
# DeepSeek配置
# =========================

load_dotenv()

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

print("DeepSeek连接完成")

DeepSeek连接完成


In [3]:
# =========================
# 当前路径检查
# =========================

import os

print("当前路径:", os.getcwd())
print("当前目录:", os.listdir())

当前路径: D:\Anaconda\Project\AI-Coding-Agent\agent-learning\day03
当前目录: ['.ipynb_checkpoints', 'day03', 'Day03_RAG_Basic.ipynb', 'Day03_RAG_LangGraph_Agent.ipynb', 'knowledge', 'vector_store']


In [4]:
# =========================
# 批量加载知识库
# =========================

from pathlib import Path
from langchain_community.document_loaders import TextLoader

documents = []

knowledge_path = Path("knowledge")

# 遍历知识库目录中的Markdown文件
for file in knowledge_path.glob("*.md"):

    loader = TextLoader(
        str(file),
        encoding="utf-8"
    )

    # 加载文档并加入知识库列表
    documents.extend(loader.load())

print("知识文档数量:", len(documents))

C:\Users\admin\AppData\Local\Temp\ipykernel_26708\3135852709.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


知识文档数量: 2


In [5]:
# =========================
# 文档切片
# =========================

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("切片数量:", len(chunks))

切片数量: 13


In [7]:
# ==========================================
# 检查模型权重文件大小
# ==========================================

from pathlib import Path

model_path = Path("../models/bge-small-zh-v1.5")
weight_path = model_path / "model.safetensors"

print(f"模型目录存在：{model_path.exists()}")
print(f"权重文件存在：{weight_path.exists()}")
print(f"权重文件大小：{weight_path.stat().st_size / 1024 / 1024:.2f} MB")

模型目录存在：True
权重文件存在：True
权重文件大小：91.39 MB


In [8]:
# ==============================
# 查看当前 Embedding 模型配置
# ==============================

print(f"当前Embedding模型：{embedding_model.model_name}")

当前Embedding模型：../models/bge-small-zh-v1.5


In [9]:
# =========================
# 创建FAISS向量库
# =========================

from langchain_community.vectorstores import FAISS


vector_store = FAISS.from_documents(
    chunks,
    embedding_model
)

print("FAISS向量库创建完成")

# =========================
# 保存FAISS索引
# =========================

vector_store.save_local(
    "day03/vector_store"
)

print("FAISS索引保存完成")

FAISS向量库创建完成
FAISS索引保存完成


In [10]:
# =========================
# 加载FAISS知识库
# =========================

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import os

model_path = "../models/bge-small-zh-v1.5"

print("模型存在:", os.path.exists(model_path))

embedding_model = HuggingFaceEmbeddings(
    model_name=model_path,
    model_kwargs={
        "local_files_only": True
    }
)

# FAISS知识库加载完成
vector_store = FAISS.load_local(
    "./vector_store",
    embedding_model,
    allow_dangerous_deserialization=True
)

print("FAISS知识库加载完成")

模型存在: True


Loading weights: 100%|██████████| 71/71 [00:00<00:00, 6519.74it/s]

FAISS知识库加载完成


In [11]:
# =========================
# RAG检索工具
# =========================

def retrieve_knowledge(query, k=3):
    """
    根据问题检索知识库
    """

    results = vector_store.similarity_search(
        query,
        k=k
    )

    context = ""

    for doc in results:
        context += "\n来源:" + doc.metadata["source"]
        context += "\n" + doc.page_content

    return context


print("RAG检索工具创建完成")

RAG检索工具创建完成


In [12]:
# =========================
# RAG检索测试
# =========================

context = retrieve_knowledge(
    "Unity背包系统如何设计"
)

print(context)


来源:day03\knowledge\Inventory.md
# Unity背包系统


## InventoryManager

InventoryManager负责玩家背包管理。


主要功能：

- 添加物品
- 删除物品
- 查询物品
- 保存背包
来源:day03\knowledge\Unity.md
# Unity开发规范


## 项目架构

Unity项目通常采用模块化设计。


主要模块：

- Manager管理模块
- Data数据模块
- UI交互模块
- Controller控制模块
来源:day03\knowledge\Inventory.md
- ID
- 名称
- 图标
- 类型
- 最大堆叠数量


## InventorySlot

InventorySlot表示背包格子。


包含：

- Item
- Count


In [13]:
# =========================
# Agent状态定义
# =========================

from typing import TypedDict


class AgentState(TypedDict):

    # 用户任务
    task: str

    # 需求分析结果
    requirements: str

    # RAG知识上下文
    rag_context: str

    # RAG检索来源
    rag_sources: list[str]

    # 架构设计结果
    architecture: str

    # 代码结果
    code: str

    # 审查结果
    review: str

    # 最终报告
    final_report: str


print("AgentState更新完成")

AgentState更新完成


In [14]:
# =========================
# RAG Agent
# =========================

def rag_agent(state: AgentState):

    print("[RAG Agent]开始执行")

    results = vector_store.similarity_search(
        state["task"],
        k=3
    )

    context = ""
    sources = []

    for doc in results:

        source = doc.metadata["source"]

        sources.append(source)

        context += "\n来源:" + source
        context += "\n" + doc.page_content
        context += "\n"

    state["rag_context"] = context

    state["rag_sources"] = list(set(sources))

    print("[RAG Agent]检索Chunk数量:", len(results))

    for source in state["rag_sources"]:
        print("[RAG Agent]来源:", source)

    return state

In [15]:
# =========================
# Architecture Agent
# =========================

def architecture_agent(state: AgentState):

    print("[Architecture Agent]开始执行")

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {
                "role":"system",
                "content":
                """
你是一名Unity高级架构师。
请结合需求和企业知识库设计系统架构。
请明确引用知识库内容，如果知识库没有相关内容，不要自行编造。
输出：
1.模块划分
2.核心类设计
3.数据结构设计
4.关键实现建议
"""
            },
            {
                "role":"user",
                "content":
                f"""
用户需求：

{state["requirements"]}


企业知识库：

{state["rag_context"]}
"""
            }
        ]
    )

    state["architecture"] = response.choices[0].message.content

    print("[Architecture Agent]执行完成")

    return state

In [16]:
# =========================
# Requirement Agent
# =========================

def requirement_agent(state: AgentState):

    print("[Requirement Agent]开始执行")

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {
                "role":"system",
                "content":
                """
你是一名Unity需求分析师。
根据用户任务分析：
1.核心功能
2.技术需求
3.注意事项
"""
            },
            {
                "role":"user",
                "content":state["task"]
            }
        ]
    )

    state["requirements"] = response.choices[0].message.content

    print("[Requirement Agent]执行完成")

    return state


print("Requirement Agent创建完成")

Requirement Agent创建完成


In [17]:
# =========================
# Coder Agent
# =========================

def coder_agent(state: AgentState):

    print("[Coder Agent]开始执行")

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {
                "role":"system",
                "content":
                """
你是一名Unity高级开发工程师。

根据系统架构生成C#代码。

要求：
1.符合Unity开发规范
2.采用模块化设计
3.包含中文注释
4.输出核心代码
"""
            },
            {
                "role":"user",
                "content":
                f"""
系统架构：

{state["architecture"]}


知识库参考：

{state["rag_context"]}
"""
            }
        ]
    )

    state["code"] = response.choices[0].message.content

    print("[Coder Agent]执行完成")

    return state


print("Coder Agent创建完成")

Coder Agent创建完成


In [18]:
# =========================
# Reviewer Agent
# =========================

def reviewer_agent(state: AgentState):

    print("[Reviewer Agent]开始执行")

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {
                "role":"system",
                "content":
                """
你是一名Unity高级代码审查工程师。

请审查下面的代码。

检查：
1.Unity规范
2.C#代码质量
3.架构合理性
4.性能问题
5.潜在Bug

输出格式：

结果:
PASS 或 FAIL

问题:
列表

建议:
列表
"""
            },
            {
                "role":"user",
                "content":
                f"""
架构设计：

{state["architecture"]}

代码：

{state["code"]}"""
            }
        ]
    )

    state["review"] = response.choices[0].message.content

    print("[Reviewer Agent]执行完成")

    return state


print("Reviewer Agent创建完成")

Reviewer Agent创建完成


In [19]:
# =========================
# Report Agent
# =========================

def report_agent(state: AgentState):

    print("[Report Agent]开始执行")

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {
                "role":"system",
                "content":
                """
你是一名技术项目负责人。

请根据以下信息生成最终开发报告。

报告包含：

1.项目目标
2.需求分析
3.知识库参考
4.系统架构
5.代码实现说明
6.代码审查结果
7.后续优化建议

要求：
结构清晰，适合提交给研发团队。
"""
            },
            {
                "role":"user",
                "content":
                f"""
任务：

{state["task"]}


需求分析：

{state["requirements"]}

知识来源：

{state["rag_sources"]}

知识库：

{state["rag_context"]}


架构设计：

{state["architecture"]}


代码：

{state["code"]}


审查结果：

{state["review"]}
"""
            }
        ]
    )

    state["final_report"] = response.choices[0].message.content

    print("[Report Agent]执行完成")

    return state


print("Report Agent创建完成")

Report Agent创建完成


In [20]:
# =========================
# Workflow构建
# =========================

graph = StateGraph(
    AgentState
)

# 添加节点
graph.add_node(
    "requirement",
    requirement_agent
)

graph.add_node(
    "rag",
    rag_agent
)

graph.add_node(
    "architecture",
    architecture_agent
)

graph.add_node(
    "coder",
    coder_agent
)

graph.add_node(
    "reviewer",
    reviewer_agent
)

graph.add_node(
    "report",
    report_agent
)

# 设置入口
graph.set_entry_point(
    "requirement"
)

# 添加流程
graph.add_edge(
    "requirement",
    "rag"
)

graph.add_edge(
    "rag",
    "architecture"
)

graph.add_edge(
    "architecture",
    "coder"
)

graph.add_edge(
    "coder",
    "reviewer"
)

graph.add_edge(
    "reviewer",
    "report"
)

graph.add_edge(
    "report",
    END
)

print("RAG Workflow创建完成")

RAG Workflow创建完成


In [21]:
# =========================
# Workflow编译
# =========================

app = graph.compile()

print("Workflow编译完成")

# =========================
# Workflow测试
# =========================

result = app.invoke(
    {
        "task":"设计Unity背包系统",
        "requirements":"",
        "rag_context":"",
        "rag_sources":[],
        "architecture":"",
        "code":"",
        "review":"",
        "final_report":""
    }
)

print(result["final_report"])
print("RAG来源:", result["rag_sources"])

Workflow编译完成
[Requirement Agent]开始执行
[Requirement Agent]执行完成
[RAG Agent]开始执行
[RAG Agent]检索Chunk数量: 3
[RAG Agent]检索Chunk数量: 3
[RAG Agent]来源: day03\knowledge\Unity.md
[RAG Agent]来源: day03\knowledge\Inventory.md
[Architecture Agent]开始执行
[Architecture Agent]执行完成
[Coder Agent]开始执行
[Coder Agent]执行完成
[Reviewer Agent]开始执行
[Reviewer Agent]执行完成
[Report Agent]开始执行
[Report Agent]执行完成
好的，作为技术项目负责人，我将根据您提供的所有信息（需求分析、知识库参考、架构设计、代码实现、代码审查结果）整合并生成一份最终开发报告。

---

# 最终开发报告：Unity背包系统

**项目名称**：Unity背包系统
**版本**：v1.0
**日期**：2023-10-27
**负责人**：技术项目负责人

---

## 1. 项目目标

设计并实现一个高性能、可扩展、解耦的Unity背包系统，以支持游戏内物品的存储、管理、交互与展示。该系统需满足以下核心目标：

- **功能完整性**：实现物品的拾取/添加、删除/丢弃、使用、堆叠、移动/交换、排序、筛选与搜索等基础功能。
- **架构清晰性**：严格遵循模块化设计，划分明确的数据模块（Data Module）、管理模块（Manager Module）、UI交互模块（UI Module）和控制模块（Controller Module）。
- **性能与健壮性**：针对大量物品（100+）提供流畅的UI交互，处理所有边界情况（如数量越界、背包已满、空引用），并防止常见的内存泄漏和GC压力。
- **可扩展与可维护性**：支持通过配置文件（ScriptableObject）扩展物品类型，通过事件系统解耦各模块，并预留存档/读档接口以支持游戏进度保存。

## 2. 需求分析

基于需求分析报告，我们将核心需求归纳为以下几点：

- **核心功能**：
    - **物